# Notebook 44 — Quick Prelim-1 single query 23

Emergency retrieval for only `query-p1-23-kis`. Reuses the frozen A0 and S1 vector indexes with three fixed English visual views. It intentionally skips OPUS inference, ASR, E5, OCR, object evidence, XCLIP, QA, TRAKE, GT, leaderboard access, and submission.

In [ ]:
import os
from pathlib import Path
SELECTED_NUMERIC_IDS=(23,)
REPO_URL=os.environ.get('AIC_REPO_URL','https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF=os.environ.get('AIC_REPO_REF','TRIAGEEG')
REPO_DIR=Path(os.environ.get('AIC_REPO_DIR','/kaggle/working/AIC2026_TeamPTK_SGU'))
QUERY_INPUT=Path(os.environ.get('AIC_PRELIM1_QUERY_ROOT','/kaggle/input/datasets/irthn1311/sotuyen1-bo-de-thi'))
RAW_INPUT=Path(os.environ.get('AIC_DATA_ROOT','/kaggle/input/datasets/nadkli/dataset-aic'))
STAGE1_INPUT=Path(os.environ.get('AIC_STAGE1_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle'))
STAGE1B_INPUT=Path(os.environ.get('AIC_STAGE1B_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports'))
STAGE1E_INPUT=Path(os.environ.get('AIC_STAGE1E_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze'))
CLIP_INPUT=Path(os.environ.get('AIC_CLIP_ROOT','/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32'))
OPUS_INPUT=Path(os.environ.get('AIC_OPUS_ROOT','/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en'))
SIGLIP_INPUT=Path(os.environ.get('AIC_SIGLIP2_ROOT','/kaggle/input/datasets/irthn1311/aic2026-siglip2-base-patch16-224'))
SIGLIP_INDEX_INPUT=Path(os.environ.get('AIC_SCA1_INDEX_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-sca1-siglip2-index-v01'))
OUTPUT_ROOT=Path('/kaggle/working/prelim1_quick_query_23')
OUTPUT_ZIP=Path('/kaggle/working/prelim1_quick_query_23_bundle.zip')
print({'mode':'EMERGENCY_QUERY_23_A0_S1_3VIEW','selected_queries':SELECTED_NUMERIC_IDS,'required_inputs':{'queries':str(QUERY_INPUT),'raw':str(RAW_INPUT),'stage1':str(STAGE1_INPUT),'stage1b':str(STAGE1B_INPUT),'stage1e':str(STAGE1E_INPUT),'clip':str(CLIP_INPUT),'opus_contract_only':str(OPUS_INPUT),'siglip2':str(SIGLIP_INPUT),'siglip2_index':str(SIGLIP_INDEX_INPUT)},'disabled':{'OPUS_INFERENCE':True,'ASR':True,'E5':True,'OCR':True,'OBJECT':True,'QA':True,'TRAKE':True,'XCLIP':True,'consensus':True,'contact_sheets':True,'embeddings_export':True},'internet_required':'GIT_CLONE_OR_REFRESH; SACREMOSES_INSTALL_ONLY_IF_MISSING','model_download':False,'output_zip':str(OUTPUT_ZIP)})


In [ ]:
import importlib.util,subprocess,sys
if not (REPO_DIR/'.git').is_dir(): subprocess.run(['git','clone','--branch',REPO_REF,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(['git','fetch','--no-tags','origin',REPO_REF],cwd=REPO_DIR,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO_DIR,check=True)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip()
required=[REPO_DIR/'scripts/run_prelim1_actual_inference.py',REPO_DIR/'scripts/run_prelim1_team_candidates.py',REPO_DIR/'src/triage_eg/prelim1_team/parser.py']
missing_source=[str(path) for path in required if not path.is_file()]
if missing_source: raise RuntimeError(f'QUICK_PRELIM1_SOURCE_MISSING:{missing_source}')
missing=[]
for module,spec in [('sacremoses','sacremoses>=0.1.1')]:
    if importlib.util.find_spec(module) is None: missing.append(spec)
if missing: subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',*missing],check=True)
sys.path.insert(0,str(REPO_DIR/'src')); sys.path.insert(0,str(REPO_DIR/'scripts'))
print({'HEAD':HEAD,'dependency_install':missing or 'NOT_REQUIRED','git_status':subprocess.check_output(['git','status','--short'],cwd=REPO_DIR,text=True).strip() or 'CLEAN'})


In [ ]:
import zipfile
from run_prelim_r5_final import _find_root,_materialize_archive_root,_resolve_dataset,_resolve_mount
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1d.inputs import resolve_input_root
WORK=Path('/kaggle/working/prelim1_quick_inputs'); WORK.mkdir(parents=True,exist_ok=True)
query_mount=_resolve_mount(QUERY_INPUT,('SOTUYEN1-bo-de-thi','sotuyen1_bo_de_thi'))
query_zips=sorted(query_mount.rglob('SOTUYEN1-bo-de-thi.zip')); ALLOW_REPACKED=False
if len(query_zips)==1: QUERY_ZIP=query_zips[0]
elif not query_zips:
    files=sorted(path for path in query_mount.rglob('query-p1-*.txt') if path.is_file())
    if len(files)!=25: raise RuntimeError(f'QUERY_DISCOVERY_FAILED:{len(files)}')
    QUERY_ZIP=WORK/'SOTUYEN1-bo-de-thi-repacked.zip'
    with zipfile.ZipFile(QUERY_ZIP,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in files: archive.write(path,path.name)
    ALLOW_REPACKED=True
else: raise RuntimeError(f'QUERY_ZIP_AMBIGUOUS:{query_zips}')
DATASET_ROOT=_resolve_dataset(RAW_INPUT)
mounts={name:_resolve_mount(path,aliases) for name,path,aliases in [('stage1',STAGE1_INPUT,()),('stage1b',STAGE1B_INPUT,()),('stage1e',STAGE1E_INPUT,()),('clip',CLIP_INPUT,()),('opus',OPUS_INPUT,()),('siglip',SIGLIP_INPUT,()),('siglip_index',SIGLIP_INDEX_INPUT,())]}
STAGE1_ROOT=resolve_stage1_root(mounts['stage1'],search_root=None,materialize_root=WORK/'stage1')
STAGE1B_ROOT,_=resolve_input_root(mounts['stage1b'],required=('stage1b_summary.json','encoder/selected_encoder_contract.json','encoder/runtime_adapter_manifest.json'),materialize_root=WORK/'stage1b',search_root=None,archive_keyword='stage1b')
STAGE1E_ROOT,_=resolve_input_root(mounts['stage1e'],required=('stage1e_summary.json','language_path_contract.json'),materialize_root=WORK/'stage1e',search_root=None,archive_keyword='stage1e')
CLIP_ROOT,_=resolve_input_root(mounts['clip'],required=('checkpoint/ViT-B-32.pt','manifests/asset_manifest.json'),materialize_root=WORK/'clip',search_root=None,archive_keyword='clip')
OPUS_ROOT,_=resolve_input_root(mounts['opus'],required=('model/config.json','manifests/asset_manifest.json'),materialize_root=WORK/'opus',search_root=None,archive_keyword='opus')
SIGLIP_ROOT,_=resolve_input_root(mounts['siglip'],required=('model/model.safetensors','manifests/asset_manifest.json'),materialize_root=WORK/'siglip',search_root=None,archive_keyword='siglip2')
SIGLIP_INDEX_ROOT=_materialize_archive_root(mounts['siglip_index'],'index/siglip2_vectors.f16.npy',('triage_eg_sca1_siglip2_index_v01.zip',),WORK/'siglip_index')
print({'MOUNT_PREFLIGHT':'PASS','query_zip':str(QUERY_ZIP),'raw':str(DATASET_ROOT),'stage1':str(STAGE1_ROOT),'stage1b':str(STAGE1B_ROOT),'stage1e':str(STAGE1E_ROOT),'clip':str(CLIP_ROOT),'opus_contract_only':str(OPUS_ROOT),'siglip':str(SIGLIP_ROOT),'siglip_index':str(SIGLIP_INDEX_ROOT)})


In [ ]:
import json,os,shutil,time
from zipfile import ZIP_DEFLATED,ZipFile
from run_prelim1_actual_inference import _canonicalize_rows
from run_prelim1_team_candidates import _csv,_jsonl
from run_prelim_r5_final import _prediction_rows,_runtime
from triage_eg.diagnostics.sca1_siglip2_complementarity import Siglip2ExactBackend,Siglip2GroundingPipeline,Siglip2OfflineEncoder
from triage_eg.e2eg1 import SafeCoveragePipeline
from triage_eg.prelim1_team.actual import ACTUAL_SYSTEM,confidence_bucket
from triage_eg.prelim1_team.packet import CatalogResolver,write_json
from triage_eg.prelim1_team.parser import parse_prelim1_zip
from triage_eg.prelim1_team.ranking import fuse_team_frames
from triage_eg.prelim_r5.fusion import R5Settings,fuse_multiview_branch
from triage_eg.retrieval.stage1b.adapters.openai_clip_official import materialize_kaggle_expanded_tokenizer,resolve_official_asset_paths
started=time.monotonic(); shutil.rmtree(OUTPUT_ROOT,ignore_errors=True); OUTPUT_ROOT.mkdir(parents=True)
manifest=parse_prelim1_zip(QUERY_ZIP,expected_sha256='' if ALLOW_REPACKED else None) if ALLOW_REPACKED else parse_prelim1_zip(QUERY_ZIP)
queries=[query for query in manifest['queries'] if int(query['numeric_id']) in SELECTED_NUMERIC_IDS]
if [int(query['numeric_id']) for query in queries]!=list(SELECTED_NUMERIC_IDS) or any(query['task']!='KIS' for query in queries): raise RuntimeError('SELECTED_QUERY_CONTRACT_FAILED')
write_json(OUTPUT_ROOT/'selected_query_manifest.json',{'source_package':{key:manifest[key] for key in ('official_package_sha256','source_zip_sha256','content_sha256','task_counts')},'queries':queries,'gt_opened':False})
roots={'stage1':STAGE1_ROOT,'stage1b':STAGE1B_ROOT,'stage1e':STAGE1E_ROOT,'clip':CLIP_ROOT,'opus':OPUS_ROOT,'siglip':SIGLIP_ROOT}
clip_paths=resolve_official_asset_paths(CLIP_ROOT); shared_source,_=materialize_kaggle_expanded_tokenizer(clip_paths.source_root,WORK/'shared_openai_clip_source'); os.environ['AIC_OPENAI_CLIP_SOURCE_ROOT']=str(shared_source)
FAST_VIEWS={
    'TEACHER_EN':'male teacher wearing a white shirt and dark tie against a dark blue patterned background',
    'SLIDE_EN':'lecture slide with a white background, pink-purple border, and blue title bar with a globe motif and yellow and teal arrows',
    'DIAGRAM_EN':'three-level diagram connected by teal downward arrows: two boxes inside an orange container, one large dark blue center box, and two boxes inside a green container',
}
print('STAGE 1/4: loading only A0 and S1 text encoders plus frozen indexes')
runtime=_runtime(REPO_DIR,HEAD,'prelim1_quick_shared',roots,WORK)
encoder=Siglip2OfflineEncoder(SIGLIP_ROOT,device='auto',batch_size=64).load()
a0=SafeCoveragePipeline(runtime,DATASET_ROOT)
s1=Siglip2GroundingPipeline(runtime,DATASET_ROOT,grounding_encoder=encoder,grounding_backend=Siglip2ExactBackend(SIGLIP_INDEX_ROOT,stage1_root=STAGE1_ROOT))
try:
    print('STAGE 2/4: running 3 English visual views; no translation or evidence-corpus load')
    visual={}; view_rows=[]; visual_provenance=[]
    for query in queries:
        a0_by_view={}; s1_by_view={}
        for view_name,text in FAST_VIEWS.items():
            request={**query,'query_id':f'{query["query_id"]}__fast__{view_name.casefold()}','query':text,'normalized_text':text,'language':'en'}
            a0_by_view[view_name]=_prediction_rows(a0,request)
            s1_by_view[view_name]=_prediction_rows(s1,request)
            view_rows.append({'query_id':query['query_id'],'task':'KIS','view':view_name,'text':text,'language':'en','derivation':'FROZEN_MANUAL_TRANSLATION_OF_QUERY_23','gt_used':False})
        a0_rows,a0_provenance=fuse_multiview_branch(query,a0_by_view,branch='A0',settings=R5Settings())
        s1_rows,s1_provenance=fuse_multiview_branch(query,s1_by_view,branch='S1',settings=R5Settings())
        visual[query['query_id']]={'a0':a0_rows,'s1':s1_rows}
        visual_provenance.extend([*a0_provenance,*s1_provenance])
        print(f'STAGE 2/4 complete — {query["query_id"]}: 3 views x A0/S1')
    resolver=CatalogResolver(STAGE1_ROOT); candidates=[]; provenance=[]
    for query in queries:
        query_id=query['query_id']
        branch_provenance=[row for row in visual_provenance if row['query_id']==query_id]
        rows,audit=fuse_team_frames(query,a0=visual[query_id]['a0'],s1=visual[query_id]['s1'],a0_provenance=[row for row in branch_provenance if row['branch']=='A0'],s1_provenance=[row for row in branch_provenance if row['branch']=='S1'],asr_lexical=[],asr_e5=[],ocr=[],objects=[],resolver=resolver,asr_specificity=[],limit=100)
        for row in rows: row.update({'source_system':ACTUAL_SYSTEM,'confidence_bucket':confidence_bucket(row)})
        _canonicalize_rows(rows,resolver); candidates.extend(rows); provenance.extend(audit)
        print(f'STAGE 3/4: final A0/S1 fusion — {query_id} Top1={rows[0]["video_id"]}/{rows[0]["frame_id"]}')
finally:
    a0.close(); encoder.close()
candidates.sort(key=lambda row:(SELECTED_NUMERIC_IDS.index(next(q['numeric_id'] for q in queries if q['query_id']==row['query_id'])),row['candidate_rank']))
top5=[row for row in candidates if row['candidate_rank']<=5]; top10=[row for row in candidates if row['candidate_rank']<=10]; primary=[row for row in candidates if row['candidate_rank']==1]
_csv(OUTPUT_ROOT/'quick_primary.csv',primary); _csv(OUTPUT_ROOT/'quick_top5.csv',top5); _csv(OUTPUT_ROOT/'quick_top10.csv',top10); _jsonl(OUTPUT_ROOT/'quick_top100.jsonl',candidates); _jsonl(OUTPUT_ROOT/'candidate_provenance.jsonl',[*view_rows,*visual_provenance,*provenance])
lines=['# Emergency Prelim-1 Result — Query 23','','A0/S1 three-view visual retrieval only. No OPUS inference, ASR, E5, OCR, object evidence, XCLIP, GT, consensus, or submission.','']
for query in queries:
    rows=[row for row in top5 if row['query_id']==query['query_id']]; lines.extend([f'## {query["query_id"]}', '',query['normalized_text'],''])
    lines.extend(f'- #{row["candidate_rank"]} `{row["video_id"]}` frame={row["frame_id"]} t={row["video_time_sec"]:.1f}s confidence={row["confidence_bucket"]}; {row["reason_short"]}' for row in rows); lines.append('')
(OUTPUT_ROOT/'QUICK_RESULTS.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
write_json(OUTPUT_ROOT/'run_provenance.json',{'HEAD':HEAD,'query_package_sha256':manifest['official_package_sha256'],'selected_numeric_ids':SELECTED_NUMERIC_IDS,'mode':'EMERGENCY_QUERY_23_A0_S1_3VIEW','fast_views':FAST_VIEWS,'elapsed_seconds':time.monotonic()-started,'A0':'ACTIVE','S1':'ACTIVE','OPUS_INFERENCE':'DISABLED_STATIC_ENGLISH_VIEWS','ASR_E5':'DISABLED','OCR_OBJECT':'DISABLED','XCLIP':'DISABLED','gt_opened':False,'leaderboard_used':False,'submission_uploaded':False})
if len(candidates)!=100 or len(top5)!=5 or len(top10)!=10 or len(primary)!=1: raise RuntimeError('QUICK_RESULT_CARDINALITY_FAILED')
OUTPUT_ZIP.unlink(missing_ok=True)
with ZipFile(OUTPUT_ZIP,'w',ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_ROOT.rglob('*')):
        if path.is_file(): archive.write(path,path.relative_to(OUTPUT_ROOT).as_posix())
print('STAGE 4/4: QUICK_RESULTS_READY')


In [ ]:
import hashlib
print({'QUICK_RESULTS_READY':True,'queries':['query-p1-23-kis'],'primary':str(OUTPUT_ROOT/'quick_primary.csv'),'top5':str(OUTPUT_ROOT/'quick_top5.csv'),'top10':str(OUTPUT_ROOT/'quick_top10.csv'),'top100':str(OUTPUT_ROOT/'quick_top100.jsonl'),'report':str(OUTPUT_ROOT/'QUICK_RESULTS.md'),'DOWNLOAD_ZIP':str(OUTPUT_ZIP),'size_bytes':OUTPUT_ZIP.stat().st_size,'sha256':hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest(),'gt_opened':False,'submission_uploaded':False})
